In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/hospital_billing.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "isCancelled": "string",
        "isClosed": "string",
        "caseType": "string",
        "speciality": "string",
        "blocked": "string",
        "flagD": "string",
        "flagB": "string",
        "flagA": "string",
        "state": "string",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,blocked,caseType,concept:name,flagA,flagB,flagD,isCancelled,isClosed,lifecycle:transition,speciality,state,time_delta
0,A,2012-12-16 19:33:10,False,A,NEW,False,False,True,False,True,complete,A,In progress,0.0
1,A,2013-12-15 19:00:37,NA,NA,FIN,NA,NA,NA,NA,NA,complete,NA,Closed,31447648.0
2,A,2013-12-16 03:53:38,NA,NA,RELEASE,NA,NA,NA,NA,NA,complete,NA,Released,31981.0
3,A,2013-12-17 12:56:29,NA,NA,CODE OK,NA,NA,NA,NA,NA,complete,NA,NA,118971.0
4,A,2013-12-19 03:44:31,NA,NA,BILLED,NA,NA,NA,NA,NA,complete,NA,Billed,139682.0
5,AA,2012-12-26 08:50:18,False,B,NEW,False,False,True,False,True,complete,L,In progress,0.0
6,AA,2012-12-26 08:50:59,NA,NA,CHANGE DIAGN,NA,NA,NA,NA,NA,complete,NA,In progress,41.0
7,AA,2013-02-14 21:06:33,NA,NA,FIN,NA,NA,NA,NA,NA,complete,NA,Closed,4364134.0
8,AA,2013-02-14 22:12:10,NA,NA,RELEASE,NA,NA,NA,NA,NA,complete,NA,Released,3937.0
9,AA,2013-02-18 01:44:10,NA,NA,CODE OK,NA,NA,NA,NA,NA,complete,NA,NA,271920.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['blocked', 'caseType', 'concept:name', 'flagA', 'flagB', 'flagD', 'isCancelled', 'isClosed', 'lifecycle:transition', 'speciality', 'state', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
isCancelled                    categorical    event    yes    ['False', 'True']                        N/A        data_derived        
isClosed                       categorical    event    yes    ['False', 'True']                        N/A        data_derived        
caseType                       categorical    event    yes    ['A', 'B', 'C', ...]                     N/A     

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'CODE OK', 'RELEASE'}]

In [13]:
engine.branching_sets

[{'BILLED',
  'CHANGE DIAGN',
  'CHANGE END',
  'CODE ERROR',
  'CODE NOK',
  'CODE OK',
  'DELETE',
  'FIN',
  'JOIN-PAT',
  'MANUAL',
  'REJECT',
  'RELEASE',
  'REOPEN',
  'SET STATUS',
  'STORNO',
  'ZDBC_BEHAN'},
 {'CHANGE DIAGN',
  'CHANGE END',
  'CODE ERROR',
  'CODE NOK',
  'CODE OK',
  'FIN',
  'MANUAL',
  'REJECT',
  'RELEASE',
  'REOPEN',
  'STORNO'},
 {'CHANGE DIAGN',
  'CHANGE END',
  'CODE ERROR',
  'FIN',
  'MANUAL',
  'REJECT',
  'REOPEN',
  'STORNO'},
 {'CHANGE DIAGN', 'CHANGE END', 'CODE ERROR', 'REJECT', 'REOPEN', 'STORNO'}]

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/hospital_billing-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/420 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,YAWA,3,1,0,0.484404,0.449363,0.519444,0.5450,0.044444,...,0.507714,0.000000,0.207714,0.277778,0.137650,0.30,0.0,0.484801,0.0,0.0
1,0,VSOB,4,1,0,0.482811,0.521178,0.444444,0.4975,0.127273,...,0.438901,0.000000,0.138901,0.277778,0.000024,0.30,0.0,0.000000,0.0,0.0
2,0,SNUD,5,1,0,0.360682,0.285253,0.436111,0.4300,0.261538,...,0.542735,0.153846,0.138889,0.277778,0.000000,0.25,0.0,0.000000,0.0,0.0
3,0,SVAE,6,1,2,0.453444,0.515222,0.391667,0.4300,0.286667,...,0.472223,0.266667,0.055557,0.111111,0.000002,0.15,0.0,0.000000,0.0,0.0
4,0,YEVB,7,1,0,0.547971,0.626498,0.469444,0.5175,0.270588,...,0.752794,0.235294,0.167500,0.333333,0.001667,0.35,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,41,RPR,11,1,4,0.555523,0.479379,0.631667,0.6365,0.376000,...,0.140000,0.000000,0.050000,0.100000,0.000000,0.09,0.0,0.968467,0.0,1.0
366,41,UHJB,11,1,4,0.574339,0.593677,0.555000,0.5900,0.436000,...,0.313755,0.000000,0.183755,0.055556,0.311955,0.13,0.0,0.000000,0.0,0.0
367,41,GNDB,11,1,4,0.582249,0.561164,0.603333,0.6175,0.316000,...,0.337617,0.000000,0.137617,0.200000,0.075235,0.20,0.0,0.000000,0.0,0.0
368,41,CDMA,12,1,4,0.621362,0.489390,0.753333,0.7685,0.585185,...,1.211959,0.074074,0.557885,0.544444,0.571326,0.58,0.0,0.000000,0.0,0.0


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,YAWA,3,2,0,0.370187,0.507041,0.233333,0.295000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,0.0,0.00
1,0,VSOB,4,2,0,0.423120,0.512906,0.333333,0.395000,0.142857,...,0.298413,0.142857,0.055556,0.111111,0.000000,0.1000,0.0,0.475995,0.0,0.50
2,0,SNUD,5,2,0,0.494479,0.700069,0.288889,0.350000,0.250000,...,0.405556,0.250000,0.055556,0.111111,0.000000,0.1000,0.0,0.475973,0.0,0.50
3,0,SVAE,6,2,2,0.421754,0.537953,0.305556,0.355000,0.238889,...,0.377778,0.222222,0.055556,0.111111,0.000000,0.1000,0.0,0.481274,0.0,0.50
4,0,YEVB,7,2,0,0.488339,0.643344,0.333333,0.395000,0.360000,...,0.411111,0.100000,0.111111,0.222222,0.000000,0.2000,0.0,0.478985,0.0,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,49,MPV,17,4,4,0.627903,0.635668,0.620139,0.655625,0.406522,...,0.516151,0.086957,0.254195,0.097222,0.411168,0.1750,0.0,0.731229,0.0,0.75
469,49,XJY,18,4,2,0.506117,0.521262,0.490972,0.513125,0.316667,...,0.478741,0.250000,0.116241,0.111111,0.121371,0.1125,0.0,0.734039,0.0,0.75
470,49,YXWC,21,4,2,0.547286,0.587627,0.506944,0.553750,0.387037,...,0.625364,0.333333,0.167031,0.041667,0.292396,0.1250,0.0,0.731092,0.0,0.75
471,49,ZGLB,22,4,2,0.510555,0.544720,0.476389,0.516875,0.398214,...,0.749314,0.357143,0.279671,0.027778,0.531565,0.1125,0.0,0.728784,0.0,0.75


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()